## BMLL Notebook Educational Series
### Order-Level Transaction Cost Analysis (TCA)

**Market Context**: BMLL collects exchange data from over 100 global exchanges, reconstructing Level 1 (top of book),
Level 2 (market by price), and Level 3 (market by order) views. We also classify lit and dark trade flows.

**TCA Objective**: Transaction Cost Analysis decomposes the cost of executed trades into measurable components,
enabling traders and portfolio managers to assess execution quality and optimize trading strategies.

**Order-Level Analysis**: Unlike trade-level TCA (which analyzes individual executions), order-level TCA evaluates
entire orders over their full duration, accounting for:
- **Arrival price**: Best available price when the order started
- **Execution volume**: Shares filled at various prices during the order window
- **Market conditions**: Spread, depth, and volatility during the order period
- **Benchmarks**: VWAP (volume-weighted average price) and other algorithmic performance measures

This notebook illustrates order-level execution analytics in Snowflake, demonstrating how to:
1. Fetch and process market data across global exchanges
2. Handle multi-currency pricing and FX conversions
3. Calculate arrival prices and market benchmarks
4. Measure execution quality against benchmarks

In [1]:
import os
from functools import lru_cache
import sys

def install_snowflake_connector():
    os.system(sys.executable + " -m pip install snowflake-connector-python[pandas] snowflake-sqlalchemy")

@lru_cache(5)
def snowflake_connection(
    database="PROD_DERIVED_DATA_DB",
    schema="PUBLIC",
    warehouse="PROD_DERIVED_DEFAULT_WH",
    role="PROD_SALES_RO_ROLE",
    user=None,
    password=None,
    use_secret_storage=True,
    sso=None,
):
    import snowflake
    import snowflake.connector
    try:
        from snowflake.snowpark import Session as SnowparkSession
    except Exception:
        SnowparkSession = None
    try:
        import bmll2
        stor = bmll2.storage_paths()["user"]
        os.environ["SNOWFLAKE_USER"] = stor["prefix"].split("/")[1]
        os.environ["SNOWFLAKE_USE_SSO"] = "1"
    except Exception:
        pass
    if sso is None:
        sso = os.environ.get("SNOWFLAKE_USE_SSO")

    password = os.environ.get("SNOWFLAKE_PASSWORD", password)
    if not password:
        try:
            import secretstorage
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            for item in collection.search_items({"name": "default_snowflake", "system": "bmll"}):
                password = item.get_secret()
        except Exception as e:
            print(f"Error connecting to Secret Service: {e}")
    else:
        try:
            import secretstorage
            attributes = {"name": "default_snowflake", "system": "bmll"}
            label = "default_snowflake"
            secret = password
            connection = secretstorage.dbus_init()
            collection = secretstorage.get_default_collection(connection)
            if collection.is_locked():
                unlocked = collection.unlock()
                if unlocked:
                    print("Collection unlocked successfully.")
                else:
                    print("Failed to unlock the collection.")
            else:
                print("Collection is already unlocked.")
            collection.create_item(label, attributes, secret, replace=True)
        except Exception as e:
            print(e)

    if not sso:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            password=password,
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
        )
    else:
        connargs = dict(
            user=os.environ.get("SNOWFLAKE_USER", user),
            account=os.environ.get("SNOWFLAKE_ACCOUNT", "tib96089"),
            region=os.environ.get("SNOWFLAKE_REGION", "us-east-1"),
            role=role,
            database=database,
            schema=schema,
            warehouse=warehouse,
            authenticator="externalbrowser",
        )
    if SnowparkSession is not None:
        return SnowparkSession.builder.configs(connargs).create()
    return snowflake.connector.connect(**connargs)

session = snowflake_connection()
if not hasattr(session, "sql"):
    class _SFResult:
        def __init__(self, cursor):
            self._cursor = cursor

        def to_pandas(self):
            try:
                return self._cursor.fetch_pandas_all()
            except Exception:
                import pandas as pd
                rows = self._cursor.fetchall()
                cols = [c[0] for c in self._cursor.description] if self._cursor.description else []
                return pd.DataFrame(rows, columns=cols)

    class _SFSessionShim:
        def __init__(self, conn):
            self._conn = conn

        def sql(self, query):
            cur = self._conn.cursor()
            cur.execute(query)
            return _SFResult(cur)

    session = _SFSessionShim(session)

/home/bertrand/miniconda3/lib/python3.11/site-packages/bmll2/_internals/api/configure.py:116: UserWarning: BMLL configuration file (/tmp/.nb_extension_instance_vars.json) not found. BMLL data and S3 storage will not be reachable.
  warnings.warn(f"{msg} BMLL data and S3 storage will not be reachable.", UserWarning)


Initiating login request with your identity provider. A browser window should have opened for you to complete the login. If you can't see it, check existing browser windows, or your OS settings. Press CTRL+C to abort and try again...
Going to open: https://accounts.google.com/o/saml2/idp?idpid=C0433grw8&SAMLRequest=lZJRb9sgFIX%2FisWebbDTZB6KE7mJqlrL1qxxKrVvFIiDgsEFHCf%2FfthppO6hlfYAQvAdOJdzp%2FNTLYMjN1ZolYE4QiDgimomVJWBbXkXpiCwjihGpFY8A2duwXw2taSWDc5bt1eP%2FK3l1gX%2BImXxcJCB1iisiRUWK1Jzix3Fm%2FzXCicRwo3RTlMtwQfJ1wpiLTfOO7xKmBXe3t65BkPYdV3UjSJtKpgghCD6AT3VI9%2Bu%2FMnX9AkfQ3TT857w%2BPrd261Qly%2F4ytbrBbL4vizX4fphU4Igv1pdaGXbmpsNN0dB%2BfZxdTFgvYPXWsqwnxyn%2B8gq3e0kOXCq66Z1%2FsbIr%2BCOMyh1JXzRxTIDzUGw8nRcVvmpcm%2FrXKan1fnhWajd%2FdPqQBg5kuqlaOk2%2FSlE94eC4OmaatKnWljb8kL1WTq%2FhZJJiNIQfS%2FjMU4meDSO4vHkBQRLn6VQxA3Kq2FCqW6Vs1GldSX54E%2FDPrgECtbM%2FRAsW6Cb0agyXQouDYKHN83sf8qewo%2FK9z777b%2B%2BWK61FPQc3GlTE%2Fd5MnEUDzuChbsBxbwmQuaMGW6tT0hK3S0MJ863szMtB3B2efXfhp79BQ%3D%3D&RelayState=ver%3A3-hint%

In [2]:
ticker = 'AAPL'
opol = 'XNAS'
start_date = '2024-09-26'

## Part 1: Key Concepts & Order Framework

**Order Duration**: The time window from order submission (st) to order completion (et). During this window:
- We measure how much volume was available for execution
- We calculate what benchmark prices (VWAP, TWAP) were available
- We compare actual execution prices against these benchmarks

**Arrival Price**: The CBBO (consolidated best bid/offer) at the moment the order started.
This is the "fair market value" benchmark - how tight was execution relative to market prices at start time.

**Execution Window**: All trades matching the order criteria (ticker, time range, price limit) during the order period.
We track what fraction of available volume was executed ("market impact") and at what prices.

**Global Markets**: Complication from international trading:
- Different exchanges have different trading hours (UTC offsets)
- Currency conversions needed for non-USD securities
- CBBO calculated from primary exchange plus dark/lit venues

In [3]:
import os
from datetime import datetime
import pandas as pd
import numpy as np

In [4]:
# Expect an existing Snowpark session named `session` in Snowflake worksheets.

pd.options.display.max_columns = None
pd.options.display.max_rows = 200

## Part 2: Sample Order Data Definition

Below we define sample orders for TCA analysis. Each order represents:
- **Ticker**: Security symbol
- **OPOL**: Primary exchange (Market Identifier Code)
- **Side**: Buy (B) or Sell (S)
- **Price**: Actual execution price
- **Limit**: Price limit for the order (orders only execute at/better than limit)
- **Shares**: Quantity of shares
- **st/et**: Start and end timestamps (order submission to completion)

In [5]:
# Example 1: Multi-asset order
data = {
    'Ticker': ['VOD', 'AAPL'],
    'OPOL': ['XLON', 'XNAS'],
    'Side': ['B', 'B'],
    'Price': [75.60, 227.70],
    'Limit': [76.00, 229.00],
    'Shares': [10000, 15000],
    'st': ['2024-09-26 10:00:00.000', '2024-09-26 10:00:00.000'],
    'et': ['2024-09-26 13:00:00.000', '2024-09-26 13:00:00.000']
}
orders = pd.DataFrame(data)

In [6]:
# Example 2: International security (GBP-traded on LSE)
data = {
    'Ticker': ['DGE'],
    'OPOL': ['XLON'],
    'Side': ['B'],
    'Price': [2615.5133],
    'Limit': [2614],
    'Shares': [5000],
    'st': ['10/4/2024  11:13:34.941'],
    'et': ['10/4/2024  11:24:35.154']
}
orders2 = pd.DataFrame(data)

In [7]:
orders_list = orders.values.tolist()

## Part 3: Data Retrieval and Processing Pipeline

The functions below handle:
- **Market data fetching** - Retrieving CBBO quotes and trades from BMLL
- **Time conversions** - Converting between local exchange times and UTC
- **Currency handling** - Managing multi-currency orders and FX conversions
- **Data aggregation** - Preparing data for TCA metric calculations

### Time and Currency Conversions

Different global exchanges operate in different time zones. BMLL timestamps are UTC, but user inputs
may be in local exchange time. These functions handle:
- Converting local times to UTC for proper market data alignment
- Determining the correct currency for each trade
- Applying FX conversion rates for international securities

In [8]:
def _apply_fx_rates(trades: pd.DataFrame, date: str) -> pd.DataFrame:
    """
    Convert trade prices to order currency using FX rates.

    For orders in non-USD currencies, converts trade prices from the execution currency
    to the order's target currency using daily FX rates. For example, a GBP-denominated
    order executed on LSE (priced in GBP) gets converted to GBP at the execution price.

    Parameters:
        trades (pd.DataFrame): Trade data with 'Price', 'CurrencyCode', 'TradeCurrency' columns
        date (str): Trading date for FX rate lookup

    Returns:
        pd.DataFrame: Input DataFrame with adjusted 'Price' column (converted to trade currency)

    Error handling:
        - Missing FX rates default to 1.0 (no conversion)
        - Exceptions are printed; function returns original prices on failure
    """
    try:
        curr_list = list((trades[trades['TradeCurrency'] != trades['CurrencyCode']]['CurrencyCode'] + 
                          trades[trades['TradeCurrency'] != trades['CurrencyCode']]['TradeCurrency']).unique())
        if curr_list:
            rates = get_fx(curr_list, from_date=date, to_date=date).transpose().reset_index()
            rates.columns = ['CurrTicker', 'Rate']
            trades = trades.merge(rates, how='left')
            trades['Rate'].fillna(1, inplace=True)
            trades['Price'] = trades['Price'] * trades['Rate']
    except Exception as e:
        print(f"FX rate adjustment failed: {e}")
    return trades

def _get_closest_record(df: pd.DataFrame, timestamp_col: str, target_timestamp: pd.Timestamp):
    """
    Find the record with timestamp closest to and not after target_timestamp.

    Used to find arrival price (CBBO at order start time) and other point-in-time lookups.
    Finds the most recent quote before or at the target time.

    Parameters:
        df (pd.DataFrame): Data to search (e.g., CBBO quotes)
        timestamp_col (str): Name of timestamp column
        target_timestamp (pd.Timestamp): Target time for lookup

    Returns:
        pd.Series or None: Row with closest timestamp, or None if no valid record exists
    """
    closest_idx = (df[timestamp_col] - target_timestamp).abs().idxmin()
    if df.loc[closest_idx, timestamp_col] <= target_timestamp:
        return df.iloc[closest_idx]
    elif closest_idx > 0:
        return df.iloc[closest_idx - 1]
    else:
        return None

def to_iso_format(date_str):
    """
    Parse various date/time formats and return ISO format 'YYYY-MM-DD HH:MM:SS'.

    Handles multiple common formats to enable flexible user input (e.g., '10/4/2024 11:13:34.941'
    from Excel exports, 'YYYY-MM-DD HH:MM:SS.ffffff' from APIs, etc.).

    Parameters:
        date_str (str): Date string in one of supported formats

    Returns:
        str: Normalized date in 'YYYY-MM-DD HH:MM:SS' format

    Raises:
        ValueError: If date string doesn't match any supported format
    """
    date_str = date_str.strip()
    for fmt in ("%m/%d/%Y %H:%M:%S.%f", "%Y-%m-%d %H:%M:%S.%f", "%Y-%m-%d %H:%M:%S"):
        try:
            return datetime.strptime(date_str, fmt).strftime("%Y-%m-%d %H:%M:%S")
        except ValueError:
            continue
    raise ValueError(f"Unsupported date format: {date_str}")

## Part 4: Order-Level TCA Calculation & Execution

### Algorithm Overview

The main TCA function orchestrates the complete analysis for a single order:

1. **Parse and normalize inputs** - Convert user timestamps and parameters to standard formats
2. **Look up security** - Map ticker + exchange to BMLL instrument identifier
3. **Determine time zones** - Convert local times to UTC using exchange calendar data
4. **Fetch market data** - Retrieve trades and CBBO quotes for the order period
5. **Calculate benchmarks** - Compute VWAP (volume-weighted average price) for the order window
6. **Measure execution** - Compare actual execution prices against benchmarks
7. **Handle currency** - Convert multi-currency prices using FX rates
8. **Return metrics** - DataFrame with arrival price, VWAP, execution volume, etc.

The function handles edge cases like missing data, non-US markets, and currency conversions.

In [9]:
def get_trades(mic, ticker, date, stt):
    try:
        sql = f"""SELECT "TICKER" AS "Ticker", "MIC" AS "MIC", "LOCAL_TRADE_TIMESTAMP" AS "LocalTradeTimestamp", "PRICE" AS "Price", "SIZE" AS "Size", "CURRENCY_CODE" AS "CurrencyCode", "PRINTABLE" AS "Printable", "EXECUTION_VENUE" AS "ExecutionVenue", "BMLL_TRADE_TYPE" AS "BMLLTradeType", "MARKET_STATE" AS "MarketState", "ORIGINAL_TRADE_TYPE" AS "OriginalTradeType" FROM "PROD_L2_CBBO_DB"."PUBLIC"."VW_TRADES" WHERE "MIC" = '{mic}' AND "TICKER" = '{ticker}' AND "TRADE_DATE" = '{date}' AND "LOCAL_TRADE_TIMESTAMP" >= '{stt}' ORDER BY "LOCAL_TRADE_TIMESTAMP" """
        return session.sql(sql).to_pandas()
    except Exception as e:
        print(f"Error fetching trades for {ticker} on {date} at {mic}: {e}")
        return None

In [10]:
def _get_reference_listing_id(ticker, mic, date):
    sql = f"""SELECT "LISTING_ID" AS "ListingId" FROM "PROD_DERIVED_DATA_DB"."PUBLIC"."REFERENCE" WHERE "TICKER" = '{ticker}' AND "MIC" = '{mic}' AND "IS_ALIVE" = TRUE AND "REFERENCE_DATE" = '{date}' QUALIFY ROW_NUMBER() OVER (ORDER BY "LISTING_ID") = 1"""
    df = session.sql(sql).to_pandas()
    if df.empty:
        raise ValueError("No listing id found for ticker/mic/date.")
    return int(df.iloc[0]["ListingId"])

In [11]:
def _get_reference_primary_currency(ticker, opol, date):
    sql = f"""SELECT "CURRENCY_CODE" AS "CurrencyCode" FROM "PROD_DERIVED_DATA_DB"."PUBLIC"."REFERENCE" WHERE "TICKER" = '{ticker}' AND "OPOL" = '{opol}' AND "IS_PRIMARY" = TRUE AND "REFERENCE_DATE" = '{date}' QUALIFY ROW_NUMBER() OVER (ORDER BY "LISTING_ID") = 1"""
    df = session.sql(sql).to_pandas()
    if df.empty:
        return None
    return df.iloc[0]["CurrencyCode"]

In [12]:
def _get_reference_mics(ticker, opol, date):
    sql = f"""SELECT DISTINCT "MIC", "TICKER" FROM "PROD_DERIVED_DATA_DB"."PUBLIC"."REFERENCE" WHERE "OPOL" = '{opol}' AND "TICKER" = '{ticker}' AND "REFERENCE_DATE" = '{date}'"""
    df = session.sql(sql).to_pandas()
    if df.empty:
        return []
    return list(zip(df["MIC"].tolist(), df["TICKER"].tolist()))

In [13]:
def _get_cbbo(mic, ticker, date, stt):
    sql = f"""SELECT "TRADE_DATE" AS "TradeDate", "EVENT_TIMESTAMP" AS "EventTimestamp", "LOCAL_TIMESTAMP" AS "LocalTimestamp", "BID_PRICE1" AS "BidPrice1", "BID_QUANTITY1" AS "BidQuantity1", "ASK_PRICE1" AS "AskPrice1", "ASK_QUANTITY1" AS "AskQuantity1", "MIC" AS "MIC", "TICKER" AS "Ticker", "BMLL_OBJECT_ID" AS "ListingId", "CURRENCY_CODE" AS "CurrencyCode" FROM "PROD_L2_CBBO_DB"."PUBLIC"."VW_CBBO_EQUITY" WHERE "MIC" = '{mic}' AND "TICKER" = '{ticker}' AND "TRADE_DATE" = '{date}' AND "LOCAL_TIMESTAMP" >= '{stt}' ORDER BY "LOCAL_TIMESTAMP" """
    return session.sql(sql).to_pandas()

In [14]:
def get_tca(Ticker, OPOL, Side, Price, Limit, Shares, st, et):
    try:
        st = to_iso_format(st)
        et = to_iso_format(et)
        date = st[:10]

        listing_id = _get_reference_listing_id(Ticker, OPOL, date)

        us_mics = ['XNAS', 'XBOS', 'XPSX', 'XNYS', 'XASE', 'ARCX', 'BATY', 'BATS',
                   'EDGA', 'EDGX', 'XCIS', 'XCHI', 'IEXG', 'MEMX', 'EPRL']

        if OPOL in us_mics:
            args = [('@SIP', Ticker)]
        else:
            args = _get_reference_mics(Ticker, OPOL, date)

        trades_list = []
        for mic, ticker in args:
            trade_data = get_trades(mic, ticker, date, st)
            if trade_data is not None:
                trades_list.append(trade_data)

        if trades_list:
            trades = pd.concat(trades_list, ignore_index=True).sort_values('LocalTradeTimestamp')
        else:
            return None

        if OPOL in us_mics:
            trades['TradeCurrency'] = 'USD'
        else:
            primary_ccy = _get_reference_primary_currency(Ticker, OPOL, date)
            trades['TradeCurrency'] = primary_ccy or trades['CurrencyCode']
        trades['CurrTicker'] = trades['CurrencyCode'] + trades['TradeCurrency']

        trades_interval = trades[(trades['LocalTradeTimestamp'] >= st) & (trades['LocalTradeTimestamp'] <= et)]
        trades_within_limit = trades_interval[trades_interval['Price'] <= Limit]

        inter_volume = trades_interval['Size'].sum()
        vwap = (trades_interval['Size'] * trades_interval['Price']).sum() / inter_volume if inter_volume > 0 else None

        int_high = trades_interval['Price'].max()
        int_low = trades_interval['Price'].min()

        inter_volume_within_lmt = trades_within_limit['Size'].sum()
        vwap_within_lmt = (trades_within_limit['Size'] * trades_within_limit['Price']).sum() / inter_volume_within_lmt if inter_volume_within_lmt > 0 else None

        market_data_mic = '@SIP' if OPOL in us_mics else OPOL
        quotes = _get_cbbo(market_data_mic, Ticker, date, st)

        target_st = pd.to_datetime(st)
        target_et = pd.to_datetime(et)

        closest_quote_arr = _get_closest_record(quotes, 'LocalTimestamp', target_st)

        if closest_quote_arr is None:
            if quotes is not None and not quotes.empty:
                closest_quote_arr = quotes.iloc[0]
            else:
                return None

        mid_arr = (closest_quote_arr['AskPrice1'] + closest_quote_arr['BidPrice1']) / 2
        bid_arr = closest_quote_arr['BidPrice1']
        ask_arr = closest_quote_arr['AskPrice1']
        spr_arr = ask_arr - bid_arr
        bid_arr_size = closest_quote_arr['BidQuantity1']
        ask_arr_size = closest_quote_arr['AskQuantity1']

        closest_trade_arr = _get_closest_record(trades, 'LocalTradeTimestamp', target_st)
        trade_arr = closest_trade_arr['Price'] if closest_trade_arr is not None else None

        arr_bps = ((Price - mid_arr) / Price) * 10000 if Price else None

        closest_quote_end = _get_closest_record(quotes, 'LocalTimestamp', target_et)
        mid_end = (closest_quote_end['AskPrice1'] + closest_quote_end['BidPrice1']) / 2 if closest_quote_end is not None else None

        def get_price_at_offset(offset_minutes):
            target_time = target_et + pd.Timedelta(minutes=offset_minutes)
            rec = _get_closest_record(trades, 'LocalTradeTimestamp', target_time)
            if rec is not None:
                return rec['Price'], rec['LocalTradeTimestamp']
            else:
                return None, None

        price_plus_1, price_plus_1_time = get_price_at_offset(1)
        price_plus_30, price_plus_30_time = get_price_at_offset(30)

        trades_after_st = trades[trades['LocalTradeTimestamp'] >= st].copy()
        trades_after_st['cumul_size'] = trades_after_st['Size'].cumsum()

        def calculate_pwp(trades_df, shares, rate):
            filtered = trades_df[trades_df['cumul_size'] <= shares / rate]
            if filtered.empty:
                filtered = trades_df.head(1)
            total_size = filtered['Size'].sum()
            weighted_avg_price = (filtered['Size'] * filtered['Price']).sum() / total_size if total_size > 0 else None
            last_time = filtered.iloc[-1]['LocalTradeTimestamp'] if not filtered.empty else None
            return weighted_avg_price, last_time

        pwp_rates = {
            'PWP_1': 0.01,
            'PWP_5': 0.05,
            'PWP_7_5': 0.075,
            'PWP_10': 0.10,
            'PWP_15': 0.15,
            'PWP_20': 0.20,
            'PWP_25': 0.25,
            'PWP_30': 0.30
        }

        pwp_results = {k: calculate_pwp(trades_after_st, Shares, rate) for k, rate in pwp_rates.items()}

        gmt_open = trades[trades['MarketState'] == 'OPENING_AUCTION']['Price'].iloc[0] if not trades[trades['MarketState'] == 'OPENING_AUCTION'].empty else None
        gmt_close = trades[trades['MarketState'] == 'CLOSING_AUCTION']['Price'].iloc[0] if not trades[trades['MarketState'] == 'CLOSING_AUCTION'].empty else None

        output = pd.DataFrame({
            'Ticker': [Ticker],
            'OPOL': [OPOL],
            'Side': [Side],
            'Price': [Price],
            'Limit': [Limit],
            'Shares': [Shares],
            'TradeDate': [date],
            'LocalStartTimestamp': [st],
            'LocalEndTimestamp': [et],
            'InterVolume': [inter_volume],
            'InterVWAP': [vwap],
            'IntHigh': [int_high],
            'IntLow': [int_low],
            'InterVolumeWithinLimit': [inter_volume_within_lmt],
            'InterVWAPWithinLimit': [vwap_within_lmt],
            'BidArrSize': [bid_arr_size],
            'AskArrSize': [ask_arr_size],
            'ArrBps': [arr_bps],
            'BidArr': [bid_arr],
            'AskArr': [ask_arr],
            'SprArr': [spr_arr],
            'TradeArr': [trade_arr],
            'MidArr': [mid_arr],
            'MidEnd': [mid_end],
            'PricePlus1': [price_plus_1],
            'PricePlus1Time': [price_plus_1_time],
            'PricePlus30': [price_plus_30],
            'PricePlus30Time': [price_plus_30_time],
            'GmtOpen': [gmt_open],
            'GmtClose': [gmt_close],
        })

        for k, (val, time) in pwp_results.items():
            output[f'{k}'] = val
            output[f'{k}_Time'] = time

        return output
    except Exception as e:
        import traceback
        print(f"get_tca failed for {Ticker}/{OPOL}/{st}: {e}")
        traceback.print_exc()
        return None

### Execution: Single Order Analysis

Process individual orders to calculate complete TCA metrics. The function returns
arrival prices, execution benchmarks (VWAP), and comprehensive execution quality metrics.

In [15]:
# Execute TCA analysis for the first order
print(f"\n{'='*80}")
print("ORDER-LEVEL TCA ANALYSIS")
print(f"{'='*80}\n")

tca_results = get_tca(orders_list[0][0], orders_list[0][1], orders_list[0][2], orders_list[0][3], orders_list[0][4], orders_list[0][5], orders_list[0][6], orders_list[0][7])

if tca_results is not None:
    print(f"\nTicket: {orders_list[0][0]} | Venue: {orders_list[0][1]} | Order Size: {orders_list[0][5]:,.0f} shares")
    print(f"Execution Period: {orders_list[0][6]} → {orders_list[0][7]}")
    print(f"\n{'-'*80}\n")
    print(tca_results.to_string())
    print(f"\n{'='*80}\n")


ORDER-LEVEL TCA ANALYSIS




Ticket: VOD | Venue: XLON | Order Size: 10,000 shares
Execution Period: 2024-09-26 10:00:00.000 → 2024-09-26 13:00:00.000

--------------------------------------------------------------------------------

  Ticker  OPOL Side  Price  Limit  Shares   TradeDate  LocalStartTimestamp    LocalEndTimestamp   InterVolume  InterVWAP  IntHigh  IntLow  InterVolumeWithinLimit  InterVWAPWithinLimit  BidArrSize  AskArrSize    ArrBps  BidArr  AskArr  SprArr TradeArr  MidArr  MidEnd  PricePlus1             PricePlus1Time  PricePlus30            PricePlus30Time GmtOpen  GmtClose      PWP_1              PWP_1_Time      PWP_5                 PWP_5_Time    PWP_7_5               PWP_7_5_Time     PWP_10             PWP_10_Time     PWP_15                PWP_15_Time    PWP_20             PWP_20_Time     PWP_25                PWP_25_Time     PWP_30                PWP_30_Time
0    VOD  XLON    B   75.6   76.0   10000  2024-09-26  2024-09-26 10:00:00  2024-09-26 13:00:00  8.079563e+06  52.424023     75.6  0.749

In [16]:
# Sequential processing for SQL-based platforms
print("Processing all orders...")
results = [get_tca(*row) for row in orders_list]
results = [r for r in results if r is not None]
if not results:
    raise ValueError("All orders failed — check errors printed above.")
tca_results = pd.concat(results, ignore_index=True)

Processing all orders...


### Results Summary
Display aggregated TCA metrics across all analyzed orders.

In [17]:
print(f"\n{'='*80}")
print("TCA RESULTS SUMMARY")
print(f"{'='*80}\n")
print(f"Total Orders Analyzed: {len(tca_results)}")
print(f"\n{'-'*80}\n")
print(tca_results.to_string())
print(f"\n{'='*80}\n")


TCA RESULTS SUMMARY

Total Orders Analyzed: 2

--------------------------------------------------------------------------------

  Ticker  OPOL Side  Price  Limit  Shares   TradeDate  LocalStartTimestamp    LocalEndTimestamp   InterVolume   InterVWAP  IntHigh    IntLow  InterVolumeWithinLimit  InterVWAPWithinLimit  BidArrSize  AskArrSize     ArrBps  BidArr  AskArr  SprArr TradeArr   MidArr   MidEnd  PricePlus1             PricePlus1Time  PricePlus30            PricePlus30Time GmtOpen  GmtClose       PWP_1                 PWP_1_Time       PWP_5                 PWP_5_Time     PWP_7_5               PWP_7_5_Time      PWP_10             PWP_10_Time      PWP_15                PWP_15_Time     PWP_20                PWP_20_Time      PWP_25                PWP_25_Time      PWP_30                PWP_30_Time
0    VOD  XLON    B   75.6   76.0   10000  2024-09-26  2024-09-26 10:00:00  2024-09-26 13:00:00  8.079563e+06   52.424023    75.60    0.7498            8.079563e+06             52.424023      